<a href="https://colab.research.google.com/github/Madihasafi/ITC-300-Data-Visualization-/blob/main/Madiha_Safi%2C_Template_of_w4p2_Exploratory_Data_Analysis_(Data_Integration).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITC 300 – Data Visualization
## Laboratory Work: Exploratory Data Analysis (Data Integration)
### Export1 + Export2 Dataset Edition

This laboratory implements the concepts and examples from the two **Lecture 3: Data Collection Structures** presentations using **only** these two source files:

- `Export1_Columns.csv` — country export values for **2004–2007**
- `Export2_Columns.csv` — country export values for **2008–2014**

No other source dataset is required.

The laboratory follows the lecture workflow:

**Raw files → Python collections → pandas Series/DataFrame → inspection and cleaning → reshaping → integration → canonical staging  → validation → analysis-ready data**

## Learning Outcomes

After completing this laboratory, you should be able to:

1. Distinguish structured, semi-structured, and unstructured representations of the same export data.
2. Use Python **lists, tuples, sets, and dictionaries** in practical data-processing tasks.
3. Create and work with pandas **Series** and **DataFrame** objects.
4. Read CSV data and inspect dimensions, columns, data types, missing values, and duplicates.
5. Demonstrate cleaning techniques using deliberately introduced anomalies in copies of the export datasets.
6. Convert text-based numeric values to numeric types.
7. Reshape export data from wide to long format using `melt()`.
8. Combine compatible tables using `concat()`.
9. Integrate related tables using `merge()` and shared keys.
10. Compare inner, left, right, and outer joins.
11. Align schemas using column-mapping dictionaries.
12. Build a canonical staging DataFrame that preserves raw values and source metadata.
13. Validate integrated data using row counts, keys, duplicates, missingness, and data types.
14. Build a reproducible integration pipeline ready for later analysis and visualization.

## Laboratory Outline

1. Environment setup and data loading  
2. Structured, semi-structured, and unstructured representations  
3. Python collection structures  
4. pandas Series and DataFrame  
5. Reading and inspecting CSV data  
6. Cleaning concepts using controlled anomalies  
7. Reshaping wide export data to long format  
8. Merging Export1 and Export2  
9. Join types in pandas  
10. Vertical integration with `concat()`  
11. Schema mapping and column alignment  
12. Canonical staging DataFrame  
13. Structural validation  
14. End-to-end integration pipeline  
15. Practice exercises

# Part 1 – Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from IPython.display import display

print("pandas version:", pd.__version__)

pandas version: 2.2.3


## Load the two required CSV files

In [12]:
export1 = pd.read_csv("Export1_Columns.csv")
export2 = pd.read_csv("Export2_Columns.csv")

print("Export1 shape:", export1.shape)
print("Export2 shape:", export2.shape)

Export1 shape: (233, 6)
Export2 shape: (233, 9)


### Dataset structure

`Export1` contains 233 country records and export values for 2004–2007.  
`Export2` contains the same country identifiers and export values for 2008–2014.

In [13]:
display(export1.head())
display(export2.head())

print("Export1 columns:", export1.columns.tolist())
print("Export2 columns:", export2.columns.tolist())

,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,811,940,869,1076
1,Burkina Faso,BFA,548,532,673,714
2,Bangladesh,BGD,7257,9995,11745,13530
3,Bulgaria,BGR,10713,12703,16151,23263
4,Bahrain,BHR,10337,13397,15662,17314


,Country Name,Country Code,2008,2009,2010,2011,2012,2013,2014
0,Benin,BEN,1312,1039,991,1040,1154,1518,1656
1,Burkina Faso,BFA,834,1063,1727,2681,2849,3166,3551
2,Bangladesh,BGD,16181,17360,18472,25627,26887,29305,34344
3,Bulgaria,BGR,28591,21964,26836,35488,33975,37260,37845
4,Bahrain,BHR,21231,15705,17880,22945,22853,0,0


Export1 columns: ['Country Name', 'Country Code', '2004', '2005', '2006', '2007']
Export2 columns: ['Country Name', 'Country Code', '2008', '2009', '2010', '2011', '2012', '2013', '2014']


# Part 2 – Python Collection Structures

Each built-in collection type has a different role in data processing.

## 2.1 Lists – ordered and mutable

A list is appropriate for year columns because chronological order matters.

In [14]:
year_cols_1 = ["2004", "2005", "2006"]
year_cols_1.append("2007")

print("Year columns:", year_cols_1)

export1_period = export1[["Country Name", "Country Code"] + year_cols_1]
display(export1_period.head())

Year columns: ['2004', '2005', '2006', '2007']


,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,811,940,869,1076
1,Burkina Faso,BFA,548,532,673,714
2,Bangladesh,BGD,7257,9995,11745,13530
3,Bulgaria,BGR,10713,12703,16151,23263
4,Bahrain,BHR,10337,13397,15662,17314


## 2.2 Tuples – ordered and immutable

The two country identifiers form a fixed pair of merge keys.

In [15]:
key_cols = ("Country Name", "Country Code")

merged_with_tuple = pd.merge(
    export1,
    export2,
    on=list(key_cols),
    how="inner"
)

print("Fixed merge-key tuple:", key_cols)
print("Merged shape:", merged_with_tuple.shape)

Fixed merge-key tuple: ('Country Name', 'Country Code')
Merged shape: (233, 13)


## 2.3 Sets – unique values and consistency checks

Sets are useful for comparing the unique country codes present in both files.

In [16]:
codes_export1 = set(export1["Country Code"])
codes_export2 = set(export2["Country Code"])

only_in_export1 = codes_export1 - codes_export2
only_in_export2 = codes_export2 - codes_export1
common_codes = codes_export1 & codes_export2

print("Unique codes in Export1:", len(codes_export1))
print("Unique codes in Export2:", len(codes_export2))
print("Common codes:", len(common_codes))
print("Only in Export1:", only_in_export1)
print("Only in Export2:", only_in_export2)

Unique codes in Export1: 233
Unique codes in Export2: 233
Common codes: 233
Only in Export1: set()
Only in Export2: set()


## 2.4 Dictionaries – key–value mappings

A dictionary can explicitly rename fields or define analytical labels.

In [17]:
rename_map = {
    "Country Name": "country_name",
    "Country Code": "country_code"
}

export1_named = export1.rename(columns=rename_map)

print(export1_named.columns.tolist())
display(export1_named.head(3))

['country_name', 'country_code', '2004', '2005', '2006', '2007']


,country_name,country_code,2004,2005,2006,2007
0,Benin,BEN,811,940,869,1076
1,Burkina Faso,BFA,548,532,673,714
2,Bangladesh,BGD,7257,9995,11745,13530


A second dictionary maps a year to a readable label. This illustrates how dictionaries can be reused for labeling.

In [18]:
year_label_map = {
    "2004": "exports_2004",
    "2005": "exports_2005",
    "2006": "exports_2006",
    "2007": "exports_2007"
}

export1_labeled = export1.rename(columns=year_label_map)
display(export1_labeled.head(3))

,Country Name,Country Code,exports_2004,exports_2005,exports_2006,exports_2007
0,Benin,BEN,811,940,869,1076
1,Burkina Faso,BFA,548,532,673,714
2,Bangladesh,BGD,7257,9995,11745,13530


# Part 3 – pandas Analytical Structures

## 3.1 Series – one labeled column

Selecting one year creates a Series.

In [19]:
exports_2004 = export1["2004"]

print(type(exports_2004))
print(exports_2004.head())
print("Mean exports in 2004:", round(exports_2004.mean(), 2))
print("Median exports in 2004:", exports_2004.median())
print("Maximum exports in 2004:", exports_2004.max())

<class 'pandas.core.series.Series'>
0      811
1      548
2     7257
3    10713
4    10337
Name: 2004, dtype: int64
Mean exports in 2004: 329414.82
Median exports in 2004: 5125.0
Maximum exports in 2004: 11332200


A Boolean Series can identify countries whose 2004 exports exceeded 100,000.

In [20]:
high_export_mask = export1["2004"] > 100000

high_export_countries = export1.loc[
    high_export_mask,
    ["Country Name", "Country Code", "2004"]
]

print("Matching countries:", len(high_export_countries))
display(high_export_countries.head(10))

Matching countries: 49


,Country Name,Country Code,2004
11,Brazil,BRA,110744
17,Canada,CAN,381529
18,Central Europe and the Baltics,CEB,354718
19,Switzerland,CHE,202880
22,China,CHN,593264
36,Germany,DEU,999334
39,Denmark,DNK,110049
42,East Asia & Pacific (developing only),EAP,1020490
43,East Asia & Pacific (all income levels),EAS,2853990
44,Europe & Central Asia (developing only),ECA,241546


## 3.2 Creating a Series explicitly

In [21]:
sample_series = pd.Series(
    export1.loc[:5, "2004"].tolist(),
    index=export1.loc[:5, "Country Code"].tolist(),
    name="Exports 2004"
)

print(sample_series)

BEN      811
BFA      548
BGD     7257
BGR    10713
BHR    10337
BHS     3161
Name: Exports 2004, dtype: int64


## 3.3 DataFrame – two-dimensional analytical table

In [22]:
export_view = export1[["Country Name", "Country Code", "2004", "2007"]]

print(type(export_view))
print(export_view.shape)
display(export_view.head())

<class 'pandas.core.frame.DataFrame'>
(233, 4)


,Country Name,Country Code,2004,2007
0,Benin,BEN,811,1076
1,Burkina Faso,BFA,548,714
2,Bangladesh,BGD,7257,13530
3,Bulgaria,BGR,10713,23263
4,Bahrain,BHR,10337,17314


# Part 4 – Reading and Inspecting CSV Data

Immediately after reading a CSV, inspect its structure and quality.

In [23]:
for name, df in [("Export1", export1), ("Export2", export2)]:
    print(f"\n{name}")
    print("-" * 40)
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing cells:", df.isna().sum().sum())
    print("\nData types:")
    print(df.dtypes)


Export1
----------------------------------------
Shape: (233, 6)
Duplicate rows: 0
Missing cells: 0

Data types:
Country Name    object
Country Code    object
2004             int64
2005             int64
2006             int64
2007             int64
dtype: object

Export2
----------------------------------------
Shape: (233, 9)
Duplicate rows: 0
Missing cells: 0

Data types:
Country Name    object
Country Code    object
2008             int64
2009             int64
2010             int64
2011             int64
2012             int64
2013             int64
2014             int64
dtype: object


## 4.1 `head()`, `tail()`, and `sample()`

In [24]:
display(export1.head(3))
display(export1.tail(3))
display(export1.sample(3, random_state=42))

,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,811,940,869,1076
1,Burkina Faso,BFA,548,532,673,714
2,Bangladesh,BGD,7257,9995,11745,13530


,Country Name,Country Code,2004,2005,2006,2007
230,"Congo, Dem. Rep.",COD,2341,2442,2765,6540
231,Zambia,ZMB,2087,2550,4158,4722
232,Zimbabwe,ZWE,2001,1931,1957,2000


,Country Name,Country Code,2004,2005,2006,2007
84,Not classified,INX,0,0,0,0
216,Upper middle income,UMC,1682610,2071850,2514560,3048460
231,Zambia,ZMB,2087,2550,4158,4722


## 4.2 One bracket vs two brackets

In [25]:
series_result = export1["2004"]
frame_result = export1[["2005"]]
multi_column_result = export1[["Country Name", "Country Code", "2004", "2007"]]

print("Single column with one bracket:", type(series_result).__name__, series_result.shape)
print("Single column with double brackets:", type(frame_result).__name__, frame_result.shape)
print("Multiple columns:", type(multi_column_result).__name__, multi_column_result.shape)

Single column with one bracket: Series (233,)
Single column with double brackets: DataFrame (233, 1)
Multiple columns: DataFrame (233, 4)


# Part 5 – Cleaning Concepts Using Controlled Anomalies

The supplied export files are already structurally clean: they contain no missing cells and no duplicate rows.

To practice the cleaning methods taught in the lecture **without using another source file**, this section deliberately introduces a few anomalies into a **copy** of `Export1`.

The original dataset remains unchanged.

In [26]:
export1_messy = export1.copy()

# Deliberately introduce controlled anomalies for teaching purposes
export1_messy.loc[0, "2004"] = "n.a."
export1_messy.loc[1, "2005"] = "not available"
export1_messy.loc[2, "2006"] = "9,995"
export1_messy.loc[3, "Country Name"] = "  Bulgaria  "

# Add a duplicate row
export1_messy = pd.concat(
    [export1_messy, export1_messy.iloc[2:6]],
    ignore_index=True
)

display(export1_messy.head())
print("Shape with controlled anomalies:", export1_messy.shape)

/tmp/ipykernel_614/3733282898.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'n.a.' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  export1_messy.loc[0, "2004"] = "n.a."
/tmp/ipykernel_614/3733282898.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not available' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  export1_messy.loc[1, "2005"] = "not available"
/tmp/ipykernel_614/3733282898.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '9,995' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  export1_messy.loc[2, "2006"] = "9,995"


,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,n.a.,940,869,1076
1,Burkina Faso,BFA,548,not available,673,714
2,Bangladesh,BGD,7257,9995,"9,995",13530
3,Bulgaria,BGR,10713,12703,16151,23263
4,Bahrain,BHR,10337,13397,15662,17314


Shape with controlled anomalies: (237, 6)


## 5.1 Detect duplicates

In [27]:
duplicate_mask = export1_messy.duplicated()

print("Duplicate rows:", duplicate_mask.sum())
display(export1_messy.loc[duplicate_mask])

Duplicate rows: 4


,Country Name,Country Code,2004,2005,2006,2007
233,Bangladesh,BGD,7257,9995,"9,995",13530
234,Bulgaria,BGR,10713,12703,16151,23263
235,Bahrain,BHR,10337,13397,15662,17314
236,"Bahamas, The",BHS,3161,3482,3558,3888


## 5.2 Standardize text labels

In [28]:
export1_clean = export1_messy.copy()

export1_clean["Country Name"] = (
    export1_clean["Country Name"]
    .astype("string")
    .str.strip()
)

display(export1_clean.head())

,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,n.a.,940,869,1076
1,Burkina Faso,BFA,548,not available,673,714
2,Bangladesh,BGD,7257,9995,"9,995",13530
3,Bulgaria,BGR,10713,12703,16151,23263
4,Bahrain,BHR,10337,13397,15662,17314


## 5.3 Replace invalid missing-value tokens

In [29]:
missing_tokens = ["n.a.", "not available", "not avilable"]

export1_clean = export1_clean.replace(missing_tokens, pd.NA)

print(export1_clean.isna().sum())
export1_clean.head()

Country Name    0
Country Code    0
2004            1
2005            1
2006            0
2007            0
dtype: int64


,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,<NA>,940,869,1076
1,Burkina Faso,BFA,548,<NA>,673,714
2,Bangladesh,BGD,7257,9995,"9,995",13530
3,Bulgaria,BGR,10713,12703,16151,23263
4,Bahrain,BHR,10337,13397,15662,17314


## 5.4 Convert text-based numeric values to numeric

Thousands separators must be removed before numerical conversion.

In [30]:
year_cols_export1 = ["2004", "2005", "2006", "2007"]

for col in year_cols_export1:
    export1_clean[col] = (
        export1_clean[col]
        .astype("string")
        .str.replace(",", "", regex=False)
    )
    export1_clean[col] = pd.to_numeric(export1_clean[col], errors="coerce")

print(export1_clean.dtypes)

Country Name    string[python]
Country Code            object
2004                     Int64
2005                     Int64
2006                     Int64
2007                     Int64
dtype: object


## 5.5 Handle missing numerical values

For demonstration, missing yearly export values are filled with the median of the respective year.  
In a real project, the imputation rule must be justified by the analytical context.

In [31]:
year_cols_export1 = ["2004", "2005", "2006", "2007"]

for col in year_cols_export1:
    # Ensure the column can store decimal values
    export1_clean[col] = export1_clean[col].astype("Float64")

    # Fill missing values with the column median
    median_value = export1_clean[col].median()

    export1_clean[col] = export1_clean[col].fillna(median_value)

print(export1_clean.isna().sum())
display(export1_clean.head())

Country Name    0
Country Code    0
2004            0
2005            0
2006            0
2007            0
dtype: int64


,Country Name,Country Code,2004,2005,2006,2007
0,Benin,BEN,5675.5,940.0,869.0,1076.0
1,Burkina Faso,BFA,548.0,6835.0,673.0,714.0
2,Bangladesh,BGD,7257.0,9995.0,9995.0,13530.0
3,Bulgaria,BGR,10713.0,12703.0,16151.0,23263.0
4,Bahrain,BHR,10337.0,13397.0,15662.0,17314.0


## 5.6 Remove duplicate rows

In [32]:
print("Remaining duplicates before cleaning:", export1_clean.duplicated().sum())
print("Non cleaned shape:", export1_clean.shape)
export1_clean = export1_messy.drop_duplicates()

print("Remaining duplicates:", export1_clean.duplicated().sum())
print("Cleaned shape:", export1_clean.shape)

Remaining duplicates before cleaning: 4
Non cleaned shape: (237, 6)
Remaining duplicates: 0
Cleaned shape: (233, 6)


# Part 6 – Reshaping Wide Export Data to Long Format

The source files use **wide format**, where each year is a separate column.

`melt()` converts year columns into rows.

In [33]:
export1_long = export1.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=["2004", "2005", "2006", "2007"],
    var_name="year",
    value_name="export_value"
)

print("Wide shape:", export1.shape)
print("Long shape:", export1_long.shape)
display(export1_long.head(10))

Wide shape: (233, 6)
Long shape: (932, 4)


,Country Name,Country Code,year,export_value
0,Benin,BEN,2004,811
1,Burkina Faso,BFA,2004,548
2,Bangladesh,BGD,2004,7257
3,Bulgaria,BGR,2004,10713
4,Bahrain,BHR,2004,10337
5,"Bahamas, The",BHS,2004,3161
6,Bosnia and Herzegovina,BIH,2004,3232
7,Belarus,BLR,2004,15710
8,Belize,BLZ,2004,535
9,Bermuda,BMU,2004,0


The long representation is especially useful because:
- `year` becomes an explicit variable,
- each country-year value becomes one observation,
- grouping by year or country is direct,
- chart mappings become simpler.

## 6.1 Reshape Export2 in the same way

In [34]:
export2_long = export2.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=["2008", "2009", "2010", "2011", "2012", "2013", "2014"],
    var_name="year",
    value_name="export_value"
)

print("Export2 long shape:", export2_long.shape)
display(export2_long.head())

Export2 long shape: (1631, 4)


,Country Name,Country Code,year,export_value
0,Benin,BEN,2008,1312
1,Burkina Faso,BFA,2008,834
2,Bangladesh,BGD,2008,16181
3,Bulgaria,BGR,2008,28591
4,Bahrain,BHR,2008,21231


# Part 7 – Relational Integration with `merge()`

`Export1` and `Export2` describe the same countries over different time periods.  
The correct shared entity keys are:

- `Country Name`
- `Country Code`

In [35]:
exports = pd.merge(
    export1,
    export2,
    on=["Country Name", "Country Code"],
    how="inner"
)

print("Integrated shape:", exports.shape)
display(exports.head())

Integrated shape: (233, 13)


,Country Name,Country Code,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
0,Benin,BEN,811,940,869,1076,1312,1039,991,1040,1154,1518,1656
1,Burkina Faso,BFA,548,532,673,714,834,1063,1727,2681,2849,3166,3551
2,Bangladesh,BGD,7257,9995,11745,13530,16181,17360,18472,25627,26887,29305,34344
3,Bulgaria,BGR,10713,12703,16151,23263,28591,21964,26836,35488,33975,37260,37845
4,Bahrain,BHR,10337,13397,15662,17314,21231,15705,17880,22945,22853,0,0


The result contains **233 rows × 13 columns**:

- 2 identifier columns
- 4 years from Export1
- 7 years from Export2

## 7.1 Verify the complete year range

In [36]:
year_cols_all = [str(year) for year in range(2004, 2015)]

print("Years present:", year_cols_all)
print("All required years available:", all(col in exports.columns for col in year_cols_all))

Years present: ['2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014']
All required years available: True


# Part 8 – Join Types in pandas

Because the two supplied files currently contain matching country keys, the four joins produce the same row count.  
The examples remain important because their behavior differs when unmatched keys exist.

In [37]:
join_results = {}

for how in ["inner", "left", "right", "outer"]:
    result = pd.merge(
        export1,
        export2,
        on=["Country Name", "Country Code"],
        how=how
    )
    join_results[how] = result
    print(f"{how:5s} join -> {result.shape}")

inner join -> (233, 13)
left  join -> (233, 13)
right join -> (233, 13)
outer join -> (233, 13)


## 8.1 Controlled join demonstration with unmatched keys

To make the difference visible while still using only Export1/Export2 data, create small derived subsets and remove one key from each side.

In [38]:
left_demo = export1[["Country Name", "Country Code", "2004"]].head(5).copy()
right_demo = export2[["Country Name", "Country Code", "2008"]].head(5).copy()

# Remove different rows from each derived table
left_demo = left_demo.iloc[:-1].copy()      # removes fifth country
right_demo = right_demo.iloc[1:].copy()    # removes first country

print("Left demo keys:", left_demo["Country Code"].tolist())
print("Right demo keys:", right_demo["Country Code"].tolist())

Left demo keys: ['BEN', 'BFA', 'BGD', 'BGR']
Right demo keys: ['BFA', 'BGD', 'BGR', 'BHR']


In [39]:
for how in ["inner", "left", "right", "outer"]:
    demo = pd.merge(
        left_demo,
        right_demo,
        on=["Country Name", "Country Code"],
        how=how
    )
    print(f"\n{how.upper()} JOIN")
    display(demo)


INNER JOIN


,Country Name,Country Code,2004,2008
0,Burkina Faso,BFA,548,834
1,Bangladesh,BGD,7257,16181
2,Bulgaria,BGR,10713,28591



LEFT JOIN


,Country Name,Country Code,2004,2008
0,Benin,BEN,811,NaN
1,Burkina Faso,BFA,548,834.0
2,Bangladesh,BGD,7257,16181.0
3,Bulgaria,BGR,10713,28591.0



RIGHT JOIN


,Country Name,Country Code,2004,2008
0,Burkina Faso,BFA,548.0,834
1,Bangladesh,BGD,7257.0,16181
2,Bulgaria,BGR,10713.0,28591
3,Bahrain,BHR,NaN,21231



OUTER JOIN


,Country Name,Country Code,2004,2008
0,Bahrain,BHR,NaN,21231.0
1,Bangladesh,BGD,7257.0,16181.0
2,Benin,BEN,811.0,NaN
3,Bulgaria,BGR,10713.0,28591.0
4,Burkina Faso,BFA,548.0,834.0


# Part 9 – Vertical Integration with `concat()`

After reshaping both files to the same long schema, they can be stacked vertically.

In [40]:
print("Export1 long columns:", export1_long.columns.tolist())
print("Export2 long columns:", export2_long.columns.tolist())

all_exports_long = pd.concat(
    [export1_long, export2_long],
    axis=0,
    ignore_index=True
)

print("Combined long shape:", all_exports_long.shape)
display(all_exports_long.head())
display(all_exports_long.tail())

Export1 long columns: ['Country Name', 'Country Code', 'year', 'export_value']
Export2 long columns: ['Country Name', 'Country Code', 'year', 'export_value']
Combined long shape: (2563, 4)


,Country Name,Country Code,year,export_value
0,Benin,BEN,2004,811
1,Burkina Faso,BFA,2004,548
2,Bangladesh,BGD,2004,7257
3,Bulgaria,BGR,2004,10713
4,Bahrain,BHR,2004,10337


,Country Name,Country Code,year,export_value
2558,"Yemen, Rep.",YEM,2014,0
2559,South Africa,ZAF,2014,109341
2560,"Congo, Dem. Rep.",COD,2014,10992
2561,Zambia,ZMB,2014,11071
2562,Zimbabwe,ZWE,2014,3625


## 9.1 Validate row-count addition

In [41]:
expected_rows = len(export1_long) + len(export2_long)
actual_rows = len(all_exports_long)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)

assert expected_rows == actual_rows

Expected rows: 2563
Actual rows: 2563


# Part 10 – Schema Mapping and Column Alignment

Different systems may use different labels for the same concept.  
A mapping dictionary defines canonical names.

In [42]:
canonical_rename = {
    "Country Name": "country_name",
    "Country Code": "country_code",
    "year": "period",
    "export_value": "raw_value"
}

export1_long_named = export1_long.rename(columns=canonical_rename)
display(export1_long_named.head())

,country_name,country_code,period,raw_value
0,Benin,BEN,2004,811
1,Burkina Faso,BFA,2004,548
2,Bangladesh,BGD,2004,7257
3,Bulgaria,BGR,2004,10713
4,Bahrain,BHR,2004,10337


# Part 11 – Canonical Staging DataFrame

A canonical staging table provides a stable common schema before formal preprocessing.

We will map both export files to these fields:

- `record_id`
- `source_name`
- `source_type`
- `business_key`
- `variable_name`
- `raw_value`
- `period`

In [43]:
def to_export_stage(df, source_name, year_columns):
    long_df = df.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=year_columns,
        var_name="period",
        value_name="raw_value"
    )

    stage = pd.DataFrame({
        "record_id": (
            source_name.replace(".csv", "").replace(" ", "_")
            + "_"
            + long_df["Country Code"].astype(str)
            + "_"
            + long_df["period"].astype(str)
        ),
        "source_name": source_name,
        "source_type": "CSV",
        "business_key": long_df["Country Code"],
        "variable_name": "export_value",
        "raw_value": long_df["raw_value"],
        "period": long_df["period"]
    })

    return stage

export1_stage = to_export_stage(
    export1,
    "Export1",
    ["2004", "2005", "2006", "2007"]
)

export2_stage = to_export_stage(
    export2,
    "Export2",
    ["2008", "2009", "2010", "2011", "2012", "2013", "2014"]
)

staging_df = pd.concat(
    [export1_stage, export2_stage],
    ignore_index=True
)

display(staging_df.head())
print("Staging shape:", staging_df.shape)

,record_id,source_name,source_type,business_key,variable_name,raw_value,period
0,Export1_BEN_2004,Export1,CSV,BEN,export_value,811,2004
1,Export1_BFA_2004,Export1,CSV,BFA,export_value,548,2004
2,Export1_BGD_2004,Export1,CSV,BGD,export_value,7257,2004
3,Export1_BGR_2004,Export1,CSV,BGR,export_value,10713,2004
4,Export1_BHR_2004,Export1,CSV,BHR,export_value,10337,2004


Staging shape: (2563, 7)


Canonicalization standardizes the **structure**. The raw export values remain available for later preprocessing and auditing.

# Part 12 – Structural Validation

In [44]:
print("Shape:", staging_df.shape)
print("\nColumns:")
print(staging_df.columns.tolist())

print("\nData types:")
print(staging_df.dtypes)

print("\nMissing values:")
print(staging_df.isna().sum())

print("\nDuplicate record IDs:",
      staging_df.duplicated(subset=["record_id"]).sum())

Shape: (2563, 7)

Columns:
['record_id', 'source_name', 'source_type', 'business_key', 'variable_name', 'raw_value', 'period']

Data types:
record_id        object
source_name      object
source_type      object
business_key     object
variable_name    object
raw_value         int64
period           object
dtype: object

Missing values:
record_id        0
source_name      0
source_type      0
business_key     0
variable_name    0
raw_value        0
period           0
dtype: int64

Duplicate record IDs: 0


## 12.1 Validate merge keys in the original files

In [45]:
key_cols = ["Country Name", "Country Code"]

print("Duplicate keys in Export1:",
      export1.duplicated(subset=key_cols).sum())
print("Duplicate keys in Export2:",
      export2.duplicated(subset=key_cols).sum())

Duplicate keys in Export1: 0
Duplicate keys in Export2: 0


## 12.2 Audit unmatched keys

In [46]:
left_audit = export1[key_cols].merge(
    export2[key_cols],
    on=key_cols,
    how="left",
    indicator=True
)

right_audit = export2[key_cols].merge(
    export1[key_cols],
    on=key_cols,
    how="left",
    indicator=True
)

print("Keys only in Export1:",
      (left_audit["_merge"] == "left_only").sum())
print("Keys only in Export2:",
      (right_audit["_merge"] == "left_only").sum())

Keys only in Export1: 0
Keys only in Export2: 0


## 12.3 Validate integrated dimensions

In [47]:
assert exports.shape == (233, 13)
assert exports.duplicated(subset=key_cols).sum() == 0
assert exports[key_cols].isna().sum().sum() == 0

print("Integrated table passed structural validation.")

Integrated table passed structural validation.


# Part 13 – Full pandas Integration Pipeline

This section combines the lecture stages into one reproducible workflow:

1. Read
2. Inspect
3. Validate keys
4. Merge
5. Reshape
6. Validate the final analytical structure

In [48]:
import pandas as pd

# 1. Read
e1 = pd.read_csv("Export1_Columns.csv")
e2 = pd.read_csv("Export2_Columns.csv")

# 2. Define entity keys
keys = ["Country Name", "Country Code"]

# 3. Validate before integration
assert e1.duplicated(keys).sum() == 0
assert e2.duplicated(keys).sum() == 0
assert e1[keys].isna().sum().sum() == 0
assert e2[keys].isna().sum().sum() == 0

# 4. Merge the time periods horizontally
exports_pipeline = pd.merge(
    e1,
    e2,
    on=keys,
    how="inner",
    validate="one_to_one"
)

# 5. Reshape to long format
years = [str(y) for y in range(2004, 2015)]

exports_pipeline_long = exports_pipeline.melt(
    id_vars=keys,
    value_vars=years,
    var_name="year",
    value_name="export_value"
)

exports_pipeline_long["year"] = pd.to_numeric(
    exports_pipeline_long["year"]
)

# 6. Validate final structure
assert exports_pipeline.shape == (233, 13)
assert len(exports_pipeline_long) == 233 * 11
assert exports_pipeline_long["export_value"].isna().sum() == 0

print("Wide integrated shape:", exports_pipeline.shape)
print("Long integrated shape:", exports_pipeline_long.shape)
display(exports_pipeline_long.head(10))

Wide integrated shape: (233, 13)
Long integrated shape: (2563, 4)


,Country Name,Country Code,year,export_value
0,Benin,BEN,2004,811
1,Burkina Faso,BFA,2004,548
2,Bangladesh,BGD,2004,7257
3,Bulgaria,BGR,2004,10713
4,Bahrain,BHR,2004,10337
5,"Bahamas, The",BHS,2004,3161
6,Bosnia and Herzegovina,BIH,2004,3232
7,Belarus,BLR,2004,15710
8,Belize,BLZ,2004,535
9,Bermuda,BMU,2004,0


# Part 14 – Analytical Examples from the Integrated Data

These examples demonstrate why integration and reshaping are useful before visualization.

## 14.1 Total exports by year

In [49]:
year_totals = (
    exports_pipeline_long
    .groupby("year", as_index=False)["export_value"]
    .sum()
)

display(year_totals)

,year,export_value
0,2004,76753654
1,2005,86921070
2,2006,99968673
3,2007,116648649
4,2008,132584324
5,2009,106573833
6,2010,126393309
7,2011,149132447
8,2012,150945046
9,2013,155071545


## 14.2 Average exports by year

In [50]:
year_average = (
    exports_pipeline_long
    .groupby("year", as_index=False)["export_value"]
    .mean()
    .round(2)
)

display(year_average)

,year,export_value
0,2004,329414.82
1,2005,373051.80
2,2006,429050.10
3,2007,500637.98
4,2008,569031.43
5,2009,457398.42
6,2010,542460.55
7,2011,640053.42
8,2012,647832.82
9,2013,665543.11


## 14.3 Change from 2004 to 2014

In [51]:
change_2004_2014 = exports_pipeline[
    ["Country Name", "Country Code", "2004", "2014"]
].copy()

change_2004_2014["absolute_change"] = (
    change_2004_2014["2014"] -
    change_2004_2014["2004"]
)

change_2004_2014["percent_change"] = np.where(
    change_2004_2014["2004"] != 0,
    change_2004_2014["absolute_change"] /
    change_2004_2014["2004"] * 100,
    np.nan
)

display(
    change_2004_2014
    .sort_values("absolute_change", ascending=False)
    .head(10)
)

,Country Name,Country Code,2004,2014,absolute_change,percent_change
226,World,WLD,11332200,23666400,12334200,108.842061
74,High income,HIC,9114030,17090000,7975970,87.513098
156,OECD members,OED,7875540,13952900,6077360,77.167534
155,High income: OECD,OEC,7581110,13303300,5722190,75.479580
45,Europe & Central Asia (all income levels),ECS,5377840,9756010,4378170,81.411310
115,Low & middle income,LMY,2218060,6595450,4377390,197.352191
130,Middle income,MIC,2188160,6513050,4324890,197.649623
43,East Asia & Pacific (all income levels),EAS,2853990,6940060,4086070,143.170439
216,Upper middle income,UMC,1682610,5006080,3323470,197.518736
53,European Union,EUU,4653420,7890080,3236660,69.554435


## 14.4 Top countries in 2014

In [52]:
top_2014 = exports_pipeline[
    ["Country Name", "Country Code", "2014"]
].sort_values("2014", ascending=False).head(10)

display(top_2014)

,Country Name,Country Code,2014
226,World,WLD,23666400
74,High income,HIC,17090000
156,OECD members,OED,13952900
155,High income: OECD,OEC,13303300
45,Europe & Central Asia (all income levels),ECS,9756010
53,European Union,EUU,7890080
43,East Asia & Pacific (all income levels),EAS,6940060
115,Low & middle income,LMY,6595450
130,Middle income,MIC,6513050
48,Euro area,EMU,5930130


# Part 15 – Practice Exercises

Use **only Export1 and Export2** to solve the following tasks.

### Exercise 1 – Collection structures
Create:
- a list containing all years from 2004 to 2014;
- a tuple containing the two merge-key columns;
- a set of all unique country codes;
- a dictionary that maps the original identifier columns to canonical names.

### Exercise 2 – Series
Create a Series containing 2010 export values with `Country Code` as the Series index.  
Calculate its mean, median, minimum, and maximum.

### Exercise 3 – Cleaning
Create a copy of `Export2`, deliberately change three numerical cells to text values such as `"n.a."`, `"12,345"`, and `"not available"`, then clean and convert the columns back to numeric form.

### Exercise 4 – Reshaping
Convert both export files to long format and concatenate them into one table.

### Exercise 5 – Merging
Merge Export1 and Export2 using both identifier columns and verify that the result has 233 rows and 13 columns.

### Exercise 6 – Join comparison
Using small derived subsets from Export1 and Export2, create examples where at least one country key is unmatched. Compare inner, left, right, and outer joins.

### Exercise 7 – Canonical staging
Create a staging DataFrame containing:
`record_id, source_name, source_type, business_key, variable_name, raw_value, period`.

### Exercise 8 – Validation
Report:
- input shapes,
- duplicate key counts,
- missing values,
- unmatched keys,
- merged shape,
- final long-format shape.

### Exercise 9 – Analytical preparation
Using the integrated 2004–2014 table:
- find the top 10 countries in 2014;
- calculate absolute change between 2004 and 2014;
- calculate yearly totals;
- calculate yearly averages.

## **Exercises Solution Workspace**
Add code and Markdown cells as needed.

In [57]:
# Exercise 1
# Your code here

# List of years from 2004 to 2014
years_list = list(range(2004, 2015))

# Tuple of merge keys
key_cols = ("Country Name", "Country Code")

# Set of unique country codes (from both datasets)
country_codes_set = set(export1["Country Code"]).union(
    set(export2["Country Code"])
)

# Dictionary mapping original → canonical names
column_mapping = {
    "Country Name": "country_name",
    "Country Code": "country_code"
}

print(years_list)
print(key_cols)
print(country_codes_set)
print(column_mapping)

[2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]
('Country Name', 'Country Code')
{'BRB', 'BTN', 'TWN', 'NOR', 'DNK', 'THA', 'RWA', 'LMY', 'NAM', 'GNB', 'BWA', 'SOM', 'CPV', 'HKG', 'HTI', 'OSS', 'MYS', 'SYR', 'COM', 'COD', 'SVK', 'BGD', 'GNQ', 'FRO', 'UKR', 'INX', 'CHN', 'OEC', 'ECA', 'NGA', 'SAU', 'MMR', 'MNP', 'CHL', 'FRA', 'LAO', 'GHA', 'CEB', 'JPN', 'CSS', 'IRL', 'MWI', 'DJI', 'NIC', 'PRI', 'VEN', 'MLI', 'MCO', 'BIH', 'PYF', 'SSA', 'PAN', 'KEN', 'HRV', 'NAC', 'UMC', 'BFA', 'FJI', 'GRC', 'TJK', 'WLD', 'OMN', 'LMC', 'HUN', 'GRL', 'GRD', 'LCN', 'DMA', 'SGP', 'CYP', 'TON', 'SVN', 'URY', 'ZAF', 'CHI', 'PRY', 'MDV', 'CHE', 'SYC', 'EAS', 'ISL', 'UZB', 'BRN', 'ROU', 'TUN', 'EST', 'EUU', 'NLD', 'TLS', 'EAP', 'PHL', 'TGO', 'PRK', 'LIE', 'MRT', 'PNG', 'SLE', 'STP', 'GUM', 'SMR', 'ERI', 'ECS', 'KOR', 'TCA', 'JOR', 'NCL', 'SXM', 'KWT', 'PER', 'USA', 'IND', 'GAB', 'JAM', 'LIC', 'MOZ', 'QAT', 'SWZ', 'MLT', 'ISR', 'MUS', 'FIN', 'LBR', 'CMR', 'PAK', 'ZWE', 'CIV', 'SLB', 'KGZ', 'LB

In [62]:
# Exercise 2
# Your code here

years = [str(y) for y in range(2004, 2015)]



In [63]:
export1.columns, export2.columns

(Index(['Country Name', 'Country Code', '2004', '2005', '2006', '2007'], dtype='object'),
 Index(['Country Name', 'Country Code', '2008', '2009', '2010', '2011', '2012',
        '2013', '2014'],
       dtype='object'))

In [76]:
# Create Series for 2010 exports
export_2010 = export2.set_index("Country Code")["2010"]

# Display first few values
print(export_2010.head())

# Statistics
print("Mean:", export_2010.mean())
print("Median:", export_2010.median())
print("Min:", export_2010.min())
print("Max:", export_2010.max())

Country Code
BEN      991
BFA     1727
BGD    18472
BGR    26836
BHR    17880
Name: 2010, dtype: int64
Mean: 542460.5536480687
Median: 10668.0
Min: 0
Max: 18858700


In [68]:
# Exercise 3
# Your code here

clean_df = export2.copy()

# Inject errors INTO EXISTING YEAR COLUMNS
clean_df.loc[0, "2004"] = "n.a."
clean_df.loc[1, "2005"] = "12,345"
clean_df.loc[2, "2006"] = "not available"

# Identify year columns (same logic as your notebook)
year_cols = [col for col in clean_df.columns if col.isdigit()]

# Remove commas
clean_df[year_cols] = clean_df[year_cols].replace(",", "", regex=True)

# Convert to numeric
for col in year_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# Check
print(clean_df[year_cols].head())
print(clean_df[year_cols].dtypes)

    2008   2009   2010   2011   2012   2013   2014  2004     2005  2006
0   1312   1039    991   1040   1154   1518   1656   NaN      NaN   NaN
1    834   1063   1727   2681   2849   3166   3551   NaN  12345.0   NaN
2  16181  17360  18472  25627  26887  29305  34344   NaN      NaN   NaN
3  28591  21964  26836  35488  33975  37260  37845   NaN      NaN   NaN
4  21231  15705  17880  22945  22853      0      0   NaN      NaN   NaN
2008      int64
2009      int64
2010      int64
2011      int64
2012      int64
2013      int64
2014      int64
2004    float64
2005    float64
2006    float64
dtype: object


In [69]:
# Exercise 4
# Your code here

exp1_long = export1.melt(
    id_vars=["Country Name", "Country Code"],
    var_name="year",
    value_name="export"
)

exp2_long = export2.melt(
    id_vars=["Country Name", "Country Code"],
    var_name="year",
    value_name="export"
)

combined_long = pd.concat([exp1_long, exp2_long], ignore_index=True)

print(combined_long.head())

   Country Name Country Code  year  export
0         Benin          BEN  2004     811
1  Burkina Faso          BFA  2004     548
2    Bangladesh          BGD  2004    7257
3      Bulgaria          BGR  2004   10713
4       Bahrain          BHR  2004   10337


In [70]:
# Exercise 5
# Your code here
merged = pd.merge(export1, export2, on=list(key_cols), how="inner")

print("Merged shape:", merged.shape)

Merged shape: (233, 13)


In [71]:
# Exercise 6
# Your code here
sub1 = export1.iloc[:5]
sub2 = export2.iloc[3:8]

inner = pd.merge(sub1, sub2, on="Country Code", how="inner")
left = pd.merge(sub1, sub2, on="Country Code", how="left")
right = pd.merge(sub1, sub2, on="Country Code", how="right")
outer = pd.merge(sub1, sub2, on="Country Code", how="outer")

print(inner)
print(left)
print(right)
print(outer)

  Country Name_x Country Code   2004   2005   2006   2007 Country Name_y  \
0       Bulgaria          BGR  10713  12703  16151  23263       Bulgaria   
1        Bahrain          BHR  10337  13397  15662  17314        Bahrain   

    2008   2009   2010   2011   2012   2013   2014  
0  28591  21964  26836  35488  33975  37260  37845  
1  21231  15705  17880  22945  22853      0      0  
  Country Name_x Country Code   2004   2005   2006   2007 Country Name_y  \
0          Benin          BEN    811    940    869   1076            NaN   
1   Burkina Faso          BFA    548    532    673    714            NaN   
2     Bangladesh          BGD   7257   9995  11745  13530            NaN   
3       Bulgaria          BGR  10713  12703  16151  23263       Bulgaria   
4        Bahrain          BHR  10337  13397  15662  17314        Bahrain   

      2008     2009     2010     2011     2012     2013     2014  
0      NaN      NaN      NaN      NaN      NaN      NaN      NaN  
1      NaN      NaN  

In [72]:
# Exercise 7
# Your code here
staging = combined_long.copy()

staging["record_id"] = staging.index
staging["source_name"] = "export"
staging["source_type"] = "csv"
staging["business_key"] = staging["Country Code"]
staging["variable_name"] = "export"
staging["raw_value"] = staging["export"]
staging["period"] = staging["year"]

staging = staging[[
    "record_id", "source_name", "source_type",
    "business_key", "variable_name",
    "raw_value", "period"
]]

print(staging.head())

   record_id source_name source_type business_key variable_name  raw_value  \
0          0      export         csv          BEN        export        811   
1          1      export         csv          BFA        export        548   
2          2      export         csv          BGD        export       7257   
3          3      export         csv          BGR        export      10713   
4          4      export         csv          BHR        export      10337   

  period  
0   2004  
1   2004  
2   2004  
3   2004  
4   2004  


In [73]:
# Exercise 8
# Your code here
print("export1 shape:", export1.shape)
print("export2 shape:", export2.shape)

print("Duplicates export1:",
      export1.duplicated(subset=["Country Code"]).sum())

print("Duplicates export2:",
      export2.duplicated(subset=["Country Code"]).sum())

print("Missing export1:\n", export1.isnull().sum())
print("Missing export2:\n", export2.isnull().sum())

merge_check = pd.merge(export1, export2,
                      on=list(key_cols),
                      how="outer",
                      indicator=True)

print("Unmatched:\n", merge_check["_merge"].value_counts())

print("Merged shape:", merge_check.shape)
print("Long shape:", combined_long.shape)

export1 shape: (233, 6)
export2 shape: (233, 9)
Duplicates export1: 0
Duplicates export2: 0
Missing export1:
 Country Name    0
Country Code    0
2004            0
2005            0
2006            0
2007            0
dtype: int64
Missing export2:
 Country Name    0
Country Code    0
2008            0
2009            0
2010            0
2011            0
2012            0
2013            0
2014            0
dtype: int64
Unmatched:
 _merge
both          233
left_only       0
right_only      0
Name: count, dtype: int64
Merged shape: (233, 14)
Long shape: (2563, 4)


In [74]:
# Exercise 9
# Your code here

combined_long["year"] = combined_long["year"].astype(int)

# Top 10 in 2014
top10 = combined_long[combined_long["year"] == 2014] \
    .sort_values(by="export", ascending=False) \
    .head(10)

print(top10)

# Pivot for change
pivot = combined_long.pivot_table(
    index="Country Code",
    columns="year",
    values="export"
)

pivot["change"] = pivot[2014] - pivot[2004]
print(pivot["change"].sort_values(ascending=False).head())

# Yearly totals
print(combined_long.groupby("year")["export"].sum())

# Yearly averages
print(combined_long.groupby("year")["export"].mean())

                                   Country Name Country Code  year    export
2556                                      World          WLD  2014  23666400
2404                                High income          HIC  2014  17090000
2486                               OECD members          OED  2014  13952900
2485                          High income: OECD          OEC  2014  13303300
2375  Europe & Central Asia (all income levels)          ECS  2014   9756010
2383                             European Union          EUU  2014   7890080
2373    East Asia & Pacific (all income levels)          EAS  2014   6940060
2445                        Low & middle income          LMY  2014   6595450
2460                              Middle income          MIC  2014   6513050
2378                                  Euro area          EMU  2014   5930130
Country Code
WLD    12334200.0
HIC     7975970.0
OED     6077360.0
OEC     5722190.0
ECS     4378170.0
Name: change, dtype: float64
year
2004     7675365